<a href="https://colab.research.google.com/github/mursyidproject/indonesian-multidoc-summarization-bart-minilm/blob/main/notebooks/BART_vs_MiniLM_MultiDoc_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets rouge_score nltk sentencepiece -q

  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import nltk
nltk.download('punkt')

# Load data
df = pd.read_csv("/content/data_latih_ringkasan.csv", encoding="latin-1")
print("Jumlah klaster:", len(df))
print("Kolom:", df.columns.tolist())

# Gabung 5 artikel
def gabung_artikel(row):
    teks_list = []
    for i in range(1, 6):
        judul = row.get(f"judul_{i}", "")
        isi = row.get(f"isi_{i}", "")
        if pd.notna(judul) and pd.notna(isi):
            teks_list.append(f"[JUDUL {i}] {judul} : {isi}")
    return " [SEP] ".join(teks_list)

df["source_text"] = df.apply(gabung_artikel, axis=1)
df["target_text"] = df["ringkasan_referensi"]

print("Contoh source_text (500 karakter):")
print(df.loc[0, "source_text"][:500])
print("\nContoh target_text:")
print(df.loc[0, "target_text"])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Jumlah klaster: 480
Kolom: ['cluster_id', 'topik', 'judul_1', 'isi_1', 'url_1', 'judul_2', 'isi_2', 'url_2', 'judul_3', 'isi_3', 'url_3', 'judul_4', 'isi_4', 'url_4', 'judul_5', 'isi_5', 'url_5', 'ringkasan_referensi']
Contoh source_text (500 karakter):
[JUDUL 1] AHY Targetkan Kampung Nelayan Tanjung Kait Jadi Kawasan Ekonomi Pesisir : AHY menegaskan kawasan nelayan Tanjung Kait perlu diperkuat sebagai kawasan produktif yang mendukung aktivitas ekonomi masyarakat pesisir. AHY menilai sejumlah infrastruktur lanjutan perlu diprioritaskan, antara lain breakwater, tanggul penahan abrasi, normalisasi sungai, jetty untuk tambatan kapal nelayan, SPBU nelayan, bengkel perahu, pabrik es berbasis slurry ice, serta penguatan dermaga. "Bukan hanya merapika

Contoh target_text:
Pemerintah mengumumkan program pembangunan kawasan kampung nelayan terpadu yang dipimpin langsung oleh Menteri Koordinator Infrastruktur AHY dan Menteri Kelautan Trenggono dalam kunjungan kerja pada April hingga Mei 2026 di 

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# Tokenizer mBART
tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50", src_lang="id_ID", tgt_lang="id_ID")

MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 150

# Dataset HF
dataset = Dataset.from_pandas(df[["source_text", "target_text"]])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({"train": dataset["train"], "validation": dataset["test"]})

def preprocess_function(examples):
    inputs = tokenizer(examples["source_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    targets = tokenizer(examples["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

print("Train:", len(tokenized_datasets["train"]), "Validation:", len(tokenized_datasets["validation"]))
sample = tokenized_datasets["train"][0]
print("Decoded input (200 char):", tokenizer.decode(sample["input_ids"], skip_special_tokens=True)[:200])
print("Decoded target:", tokenizer.decode([l for l in sample["labels"] if l != -100], skip_special_tokens=True)[:200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Map:   0%|          | 0/432 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Train: 432 Validation: 48
Decoded input (200 char): [JUDUL 1] Banyak Perusahaan Klaim ESG, tapi Tak Tahu Sampahnya Berakhir ke Mana : Ada sejumlah perusahaan di Indonesia yang melaporkan kinerja Environmental, Social, and Governance (ESG), tapi belum m
Decoded target: Founder Waste4Change Mohamad Bijaksana Junerosano pada 14 Maret 2026 menyatakan banyak perusahaan di Indonesia sudah membuat laporan ESG tetapi tidak tahu akhir riwayat sampah dari kegiatan ekonominya


In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=None)

model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50")

total = sum(p.numel() for p in model.parameters())
print(f"Total parameter: {total:,}")
print("Model siap!")

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Total parameter: 610,879,488
Model siap!


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import os

OUTPUT_DIR = "/content/bart_checkpoints"
final_path = "/content/bart_final_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=False,
    optim="adamw_torch",
    report_to="none",
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

trainer.train()

os.makedirs(final_path, exist_ok=True)
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print("✅ Model tersimpan di:", final_path)

Step,Training Loss
10,33.644894
20,21.899417
30,20.702541
40,19.320795
50,19.430663
60,17.262086
70,13.932402
80,14.523451
90,13.931882
100,13.913367


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model tersimpan di: /content/bart_final_model


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/bart_final_model", 'zip', "/content/bart_final_model")
files.download("/content/bart_final_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# Load data uji (sesuaikan path jika perlu)
df_uji = pd.read_csv("/content/data_uji_ringkasan.csv", encoding="latin-1")
print("Jumlah klaster uji:", len(df_uji))
print("Kolom:", df_uji.columns.tolist())

# Gabung 5 artikel (sama seperti data latih)
def gabung_artikel(row):
    teks_list = []
    for i in range(1, 6):
        judul = row.get(f"judul_{i}", "")
        isi = row.get(f"isi_{i}", "")
        if pd.notna(judul) and pd.notna(isi):
            teks_list.append(f"[JUDUL {i}] {judul} : {isi}")
    return " [SEP] ".join(teks_list)

df_uji["source_text"] = df_uji.apply(gabung_artikel, axis=1)

# Tampilkan contoh
print("\nContoh source_text (300 karakter):")
print(df_uji.loc[0, "source_text"][:300])
print("\nContoh ringkasan referensi:")
print(df_uji.loc[0, "ringkasan_referensi"])

Jumlah klaster uji: 120
Kolom: ['cluster_id', 'topik', 'judul_1', 'isi_1', 'judul_2', 'isi_2', 'judul_3', 'isi_3', 'judul_4', 'isi_4', 'judul_5', 'isi_5', 'ringkasan_referensi']

Contoh source_text (300 karakter):
[JUDUL 1] Jika Perang Timur Tengah Berakhir, Apakah Ekonomi Indonesia Otomatis Pulih? : Jika Perang Timur Tengah Berakhir, Apakah Ekonomi Indonesia Otomatis Pulih? Artikel ini adalah kolom, seluruh isi dan opini merupakan pandangan pribadi penulis dan bukan cerminan sikap redaksi. LEBIH dari seratus

Contoh ringkasan referensi:
Perang Timur Tengah memang bukan satu-satunya penentu nasib ekonomi Indonesia. Kompas lewat kolomnya memperingatkan, sejarah sudah buktikan bahwa perdamaian tanpa fondasi ekonomi yang kuat cuma jadi 'obat penenang' sementara. Sementara itu, data pertumbuhan ekonomi RI di kuartal I 2026 masih oke di 5,61% versi Detik, tapi Hermanto dari Fraksi PKS mengingatkan ini belum tentu menggembirakan karena daya beli masyarakat lesu dan rupiah masih tertekan. Ada

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "/content/bart_final_model"

print("Memuat tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="id_ID", tgt_lang="id_ID")

print("Memuat model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Pastikan model di GPU
model = model.cuda()
print("Model siap di GPU.")

Memuat tokenizer...
Memuat model...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Model siap di GPU.


In [ ]:
import torch
import re
from transformers import AutoTokenizer

# Load tokenizer lagi (jika belum ada)
tokenizer = AutoTokenizer.from_pretrained("/content/bart_final_model", src_lang="id_ID", tgt_lang="id_ID")

MAX_ARTICLE_TOKENS = 200   # Maks token per artikel (5 artikel = 1000 token)
MAX_OUTPUT_LENGTH = 250
NUM_BEAMS = 4
NO_REPEAT_NGRAM = 3

def gabung_seimbang(row, max_article_tokens=MAX_ARTICLE_TOKENS):
    teks_list = []
    for i in range(1, 6):
        judul = row.get(f"judul_{i}", "")
        isi = row.get(f"isi_{i}", "")
        if pd.notna(judul) and pd.notna(isi):
            # Gabung judul dan isi
            artikel = f"[JUDUL {i}] {judul} : {isi}"
            # Tokenize dan potong jika melebihi max_article_tokens
            tokens = tokenizer.encode(artikel, max_length=max_article_tokens, truncation=True, add_special_tokens=False)
            # Decode kembali ke teks
            artikel_potong = tokenizer.decode(tokens, skip_special_tokens=True)
            teks_list.append(artikel_potong)
    return " [SEP] ".join(teks_list)

# Terapkan pada df_uji
df_uji["source_text_balanced"] = df_uji.apply(gabung_seimbang, axis=1)

# Tampilkan panjang token rata-rata setelah pemotongan
sample_tokens = tokenizer.encode(df_uji.loc[0, "source_text_balanced"], max_length=1024, truncation=True)
print("Panjang token input (setelah seimbang):", len(sample_tokens))

# Generate ulang dengan input baru
hasil_ringkasan_baru = []
print("Generate ringkasan dengan input seimbang...")
for i, row in df_uji.iterrows():
    inputs = tokenizer(row["source_text_balanced"], max_length=1024, truncation=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=NUM_BEAMS,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            early_stopping=True,
        )
    ringkasan = tokenizer.decode(outputs[0], skip_special_tokens=True)
    hasil_ringkasan_baru.append(ringkasan)
    if (i+1) % 30 == 0:
        print(f"  {i+1}/120 selesai...")

# Bersihkan
def bersihkan_dan_rapikan(teks):
    teks = re.sub(r'\[JUDUL\s*\d+\]', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    # Ambil sampai tanda baca terakhir yang mungkin
    for punct in ['. ', '? ', '! ']:
        if punct in teks:
            last = teks.rfind(punct)
            if last > 10:
                teks = teks[:last+1]
                break
    return teks.strip()

df_uji["ringkasan_model"] = hasil_ringkasan_baru
df_uji["ringkasan_model_bersih"] = df_uji["ringkasan_model"].apply(bersihkan_dan_rapikan)

# Simpan hasil perbaikan final
df_uji.to_csv("/content/hasil_evaluasi_bart_balanced.csv", index=False, encoding="utf-8")

# Tampilkan perbaikan untuk E136 dan E236
for cid in ["E136", "E236"]:
    if cid in df_uji["cluster_id"].values:
        idx = df_uji[df_uji["cluster_id"] == cid].index[0]
        print(f"\n=== KLASTER {cid} (SETELAH PERBAIKAN) ===")
        print("REFERENSI:", df_uji.loc[idx, "ringkasan_referensi"][:250])
        print("MODEL    :", df_uji.loc[idx, "ringkasan_model_bersih"][:300])

Panjang token input (setelah seimbang): 1018
Generate ringkasan dengan input seimbang...
  30/120 selesai...
  60/120 selesai...
  90/120 selesai...
  120/120 selesai...

=== KLASTER E136 (SETELAH PERBAIKAN) ===
REFERENSI: Hari Buruh 2026 jadi momentum merefleksikan nasib pekerja di tengah otomatisasi dan ekonomi gig. Kolom di Kompas menyoroti ribuan pengemudi ojol di Surabaya yang turun jalan, tanda bahwa tantangan buruh kini bukan cuma soal upah pabrik, tapi juga mel
MODEL    : Ketua DPR Puan Maharani pada 15 April 2026 mendorong penggunaan kemasan dari bahan organik sebagai alternatif di tengah lonjakan harga plastik akibat kenaikan harga plastik yang melonjak hingga berkali-kali lipat. Dosen UIN Syarif Hidayatullah menulis kolom tentang kemandirian ekonomi Indonesia pada

=== KLASTER E236 (SETELAH PERBAIKAN) ===
REFERENSI: Bank Pembangunan Daerah (BPD) didorong naik kelas. Ketua Umum Asbanda Agus H Widodo menegaskan BPD harus bertransformasi dari sekadar penyalur dana menjadi 'orke

In [ ]:
import pandas as pd
import re

# Load file balanced
df_cek = pd.read_csv("/content/hasil_evaluasi_bart_balanced.csv")

# Pastikan kolom yang dipakai
kolom_model = "ringkasan_model_bersih" if "ringkasan_model_bersih" in df_cek.columns else "ringkasan_model"
kolom_ref = "ringkasan_referensi"

print("=== STATISTIK RINGKASAN ===\n")

# 1. Cek sisa tag [JUDUL ...]
def cek_judul_tag(teks):
    return bool(re.search(r'\[JUDUL\s*\d+\]', str(teks)))

sisa_judul = df_cek[kolom_model].apply(cek_judul_tag).sum()
print(f"1. Ringkasan dengan sisa [JUDUL ...] : {sisa_judul} dari {len(df_cek)}")

# 2. Cek akhiran tanpa tanda baca . ? !
def akhiran_lengkap(teks):
    teks = str(teks).strip()
    return teks.endswith(('.', '?', '!'))

tidak_lengkap = df_cek[~df_cek[kolom_model].apply(akhiran_lengkap)]
print(f"2. Ringkasan tanpa tanda baca akhir : {len(tidak_lengkap)} dari {len(df_cek)}")

# 3. Cek panjang
print(f"\n3. Panjang rata-rata ringkasan model  : {df_cek[kolom_model].str.len().mean():.0f} karakter")
print(f"   Panjang rata-rata ringkasan referensi: {df_cek[kolom_ref].str.len().mean():.0f} karakter")

# 4. Tampilkan contoh yang masih tidak lengkap
if len(tidak_lengkap) > 0:
    print(f"\n=== CONTOH RINGKASAN TANPA TANDA BACA AKHIR ===")
    for idx, row in tidak_lengkap.head(5).iterrows():
        print(f"\nCluster {row['cluster_id']} | Panjang: {len(str(row[kolom_model]))} karakter")
        print(f"Model    : {str(row[kolom_model])[:300]}")
        print(f"Referensi: {str(row[kolom_ref])[:200]}")

# 5. Cek ringkasan sangat pendek (<100 karakter)
pendek = df_cek[df_cek[kolom_model].str.len() < 100]
print(f"\n5. Ringkasan model sangat pendek (<100 karakter): {len(pendek)}")
if len(pendek) > 0:
    for idx, row in pendek.iterrows():
        print(f"   {row['cluster_id']} -> {str(row[kolom_model])[:150]}")

=== STATISTIK RINGKASAN ===

1. Ringkasan dengan sisa [JUDUL ...] : 0 dari 120
2. Ringkasan tanpa tanda baca akhir : 0 dari 120

3. Panjang rata-rata ringkasan model  : 573 karakter
   Panjang rata-rata ringkasan referensi: 840 karakter

5. Ringkasan model sangat pendek (<100 karakter): 0


In [ ]:
!pip install evaluate bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd, numpy as np, evaluate, nltk, re
from bert_score import score as bert_score
from google.colab import files
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)

# 1. Load data balanced
df = pd.read_csv("/content/hasil_evaluasi_bart_balanced.csv")
preds = df["ringkasan_model_bersih"].tolist()
refs  = df["ringkasan_referensi"].tolist()

# 2. Hitung ROUGE per klaster
rouge = evaluate.load("rouge")
rouge_scores = []
for p, r in zip(preds, refs):
    p_fmt = "\n".join(nltk.sent_tokenize(p))
    r_fmt = "\n".join(nltk.sent_tokenize(r))
    s = rouge.compute(predictions=[p_fmt], references=[r_fmt], use_stemmer=True)
    rouge_scores.append({k: round(v*100,2) for k,v in s.items()})

df_rouge = pd.DataFrame(rouge_scores)
df_detail = pd.concat([df[["cluster_id","topik"]], df_rouge], axis=1)
df_detail["ref_ringkasan"] = refs
df_detail["model_ringkasan"] = preds

# 3. BERTScore global (F1)
P, R, F1 = bert_score(preds, refs, lang="id", verbose=False)
prec, rec, f1 = P.mean().item(), R.mean().item(), F1.mean().item()

# 4. ROUGE global
preds_rouge = ["\n".join(nltk.sent_tokenize(p)) for p in preds]
refs_rouge  = ["\n".join(nltk.sent_tokenize(r)) for r in refs]
global_rouge = rouge.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)
global_rouge = {k: round(v*100,2) for k,v in global_rouge.items()}

# 5. DataFrame metrik global
df_metrics = pd.DataFrame([
    {"Metrik": "ROUGE-1", "Skor (%)": global_rouge["rouge1"]},
    {"Metrik": "ROUGE-2", "Skor (%)": global_rouge["rouge2"]},
    {"Metrik": "ROUGE-L", "Skor (%)": global_rouge["rougeL"]},
    {"Metrik": "BERTScore Precision", "Skor": round(prec,4)},
    {"Metrik": "BERTScore Recall", "Skor": round(rec,4)},
    {"Metrik": "BERTScore F1", "Skor": round(f1,4)},
])

# 6. Sampel terbaik & terburuk (berdasarkan ROUGE-L)
df_detail_sorted = df_detail.sort_values("rougeL", ascending=False)
terbaik = df_detail_sorted.head(5)[["cluster_id","rougeL","model_ringkasan","ref_ringkasan"]]
terburuk = df_detail_sorted.tail(5)[["cluster_id","rougeL","model_ringkasan","ref_ringkasan"]]

# 7. Tulis Excel
with pd.ExcelWriter("/content/Laporan_Evaluasi_BART.xlsx") as writer:
    df_metrics.to_excel(writer, sheet_name="Ringkasan_Metrik", index=False)
    df_detail_sorted.to_excel(writer, sheet_name="Detail_Per_Klaster", index=False)
    terbaik.to_excel(writer, sheet_name="Sampel_Terbaik", index=False)
    terburuk.to_excel(writer, sheet_name="Sampel_Terburuk", index=False)

files.download("/content/Laporan_Evaluasi_BART.xlsx")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd, numpy as np, evaluate, nltk, re, time, torch, os
from bert_score import score as bert_score
from google.colab import files
import locale
locale.getpreferredencoding = lambda: "UTF-8"

nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("/content/hasil_evaluasi_bart_balanced.csv")
preds = df["ringkasan_model_bersih"].tolist()
refs  = df["ringkasan_referensi"].tolist()

# =========================================================
# 2. HITUNG ROUGE & BERTScore (sama seperti sebelumnya)
# =========================================================
rouge = evaluate.load("rouge")
rouge_scores = []
for p, r in zip(preds, refs):
    p_fmt = "\n".join(nltk.sent_tokenize(p))
    r_fmt = "\n".join(nltk.sent_tokenize(r))
    s = rouge.compute(predictions=[p_fmt], references=[r_fmt], use_stemmer=True)
    rouge_scores.append({k: round(v*100,2) for k,v in s.items()})

df_rouge = pd.DataFrame(rouge_scores)
df_detail = pd.concat([df[["cluster_id","topik"]], df_rouge], axis=1)
df_detail["ref_ringkasan"] = refs
df_detail["model_ringkasan"] = preds

# Global ROUGE
preds_rouge = ["\n".join(nltk.sent_tokenize(p)) for p in preds]
refs_rouge  = ["\n".join(nltk.sent_tokenize(r)) for r in refs]
global_rouge = rouge.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)
global_rouge = {k: round(v*100,2) for k,v in global_rouge.items()}

# BERTScore
P, R, F1 = bert_score(preds, refs, lang="id", verbose=False)
prec, rec, f1 = P.mean().item(), R.mean().item(), F1.mean().item()

# =========================================================
# 3. UKUR EFISIENSI KOMPUTASI
# =========================================================
# Pastikan model sudah di GPU
model = None
try:
    # Cek apakah model masih di memori (dari sesi sebelumnya)
    model = AutoModelForSeq2SeqLM.from_pretrained("/content/bart_final_model").cuda()
except:
    from transformers import AutoModelForSeq2SeqLM
    model = AutoModelForSeq2SeqLM.from_pretrained("/content/bart_final_model").cuda()

tokenizer = AutoTokenizer.from_pretrained("/content/bart_final_model", src_lang="id_ID", tgt_lang="id_ID")

# Ukuran model
size_bytes = sum(p.numel() for p in model.parameters() if p.requires_grad)  # parameter trainable
total_params = sum(p.numel() for p in model.parameters())
model_size_mb = os.path.getsize("/content/bart_final_model/pytorch_model.bin") / (1024*1024) if os.path.exists("/content/bart_final_model/pytorch_model.bin") else 0

# Waktu inferensi (ambil 10 sampel acak)
sample_indices = np.random.choice(len(df), 10, replace=False)
waktu = []
for idx in sample_indices:
    teks = df.iloc[idx]["source_text_balanced"] if "source_text_balanced" in df.columns else df.iloc[idx]["source_text"]
    inputs = tokenizer(teks, max_length=1024, truncation=True, return_tensors="pt").to("cuda")
    start = time.time()
    with torch.no_grad():
        _ = model.generate(**inputs, max_length=250, num_beams=4, early_stopping=True)
    end = time.time()
    waktu.append(end-start)

rata_waktu = np.mean(waktu)
total_waktu_estimasi = rata_waktu * 120

# Penggunaan GPU
if torch.cuda.is_available():
    gpu_mem_allocated = torch.cuda.memory_allocated() / (1024**2)  # MB
    gpu_mem_reserved  = torch.cuda.memory_reserved() / (1024**2)
else:
    gpu_mem_allocated = gpu_mem_reserved = 0

# =========================================================
# 4. BUAT DATAFRAME EFISIENSI
# =========================================================
df_efisiensi = pd.DataFrame([
    {"Komponen": "Model", "Nilai": "facebook/mbart-large-50"},
    {"Komponen": "Ukuran Model (MB)", "Nilai": f"{model_size_mb:.2f}"},
    {"Komponen": "Total Parameter", "Nilai": f"{total_params:,}"},
    {"Komponen": "Trainable Parameter", "Nilai": f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"},
    {"Komponen": "Rata-rata Waktu Inferensi per Klaster (detik)", "Nilai": f"{rata_waktu:.2f}"},
    {"Komponen": "Estimasi Total Waktu 120 Klaster (detik)", "Nilai": f"{total_waktu_estimasi:.2f}"},
    {"Komponen": "GPU Memory Allocated (MB)", "Nilai": f"{gpu_mem_allocated:.2f}"},
    {"Komponen": "GPU Memory Reserved (MB)", "Nilai": f"{gpu_mem_reserved:.2f}"},
])

# =========================================================
# 5. RUMUS METRIK (SHEET BARU)
# =========================================================
# Ambil contoh satu klaster untuk perhitungan manual
contoh_pred = preds[0]
contoh_ref  = refs[0]

# Tokenisasi sederhana untuk ROUGE-1 (unigram)
def hitung_rouge1(pred, ref):
    pred_words = pred.lower().split()
    ref_words  = ref.lower().split()
    pred_counts = {}
    for w in pred_words:
        pred_counts[w] = pred_counts.get(w, 0) + 1
    ref_counts = {}
    for w in ref_words:
        ref_counts[w] = ref_counts.get(w, 0) + 1
    overlap = sum(min(pred_counts.get(w,0), ref_counts.get(w,0)) for w in set(pred_words))
    recall = overlap / len(ref_words) if ref_words else 0
    precision = overlap / len(pred_words) if pred_words else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision+recall) > 0 else 0
    return precision, recall, f1

prec1, rec1, f1_1 = hitung_rouge1(contoh_pred, contoh_ref)

df_rumus = pd.DataFrame([
    {"Metrik": "ROUGE-1 Precision", "Rumus": "overlap_unigram / panjang_ringkasan_model", "Contoh": f"{prec1:.4f}"},
    {"Metrik": "ROUGE-1 Recall", "Rumus": "overlap_unigram / panjang_ringkasan_referensi", "Contoh": f"{rec1:.4f}"},
    {"Metrik": "ROUGE-1 F1", "Rumus": "2 * Precision * Recall / (Precision + Recall)", "Contoh": f"{f1_1:.4f}"},
    {"Metrik": "ROUGE-2", "Rumus": "Sama seperti ROUGE-1 tetapi menggunakan bigram (2 kata berurutan)", "Contoh": "Dihitung otomatis oleh pustaka ROUGE"},
    {"Metrik": "ROUGE-L", "Rumus": "Menggunakan Longest Common Subsequence (LCS)", "Contoh": "Dihitung otomatis oleh pustaka ROUGE"},
    {"Metrik": "BERTScore", "Rumus": "Precision = rerata cosine similarity token model -> referensi; Recall = rerata cosine similarity token referensi -> model; F1 = 2*P*R/(P+R)", "Contoh": f"P={prec:.4f}, R={rec:.4f}, F1={f1:.4f}"},
])

# =========================================================
# 6. PERBANDINGAN BART VS MINILM (PLACEHOLDER)
# =========================================================
df_comp = pd.DataFrame([
    {"Metrik": "ROUGE-1 (%)", "BART": global_rouge["rouge1"], "MiniLM": "TBA"},
    {"Metrik": "ROUGE-2 (%)", "BART": global_rouge["rouge2"], "MiniLM": "TBA"},
    {"Metrik": "ROUGE-L (%)", "BART": global_rouge["rougeL"], "MiniLM": "TBA"},
    {"Metrik": "BERTScore F1", "BART": f"{f1:.4f}", "MiniLM": "TBA"},
    {"Metrik": "Waktu Inferensi Rata-rata (detik)", "BART": f"{rata_waktu:.2f}", "MiniLM": "TBA"},
    {"Metrik": "Ukuran Model (MB)", "BART": f"{model_size_mb:.2f}", "MiniLM": "TBA"},
    {"Metrik": "Total Parameter", "BART": f"{total_params:,}", "MiniLM": "TBA"},
])

# =========================================================
# 7. TULIS KE EXCEL
# =========================================================
with pd.ExcelWriter("/content/Laporan_Lengkap_BART.xlsx") as writer:
    # Sheet ringkasan metrik
    pd.DataFrame([
        {"Metrik": "ROUGE-1", "Skor (%)": global_rouge["rouge1"]},
        {"Metrik": "ROUGE-2", "Skor (%)": global_rouge["rouge2"]},
        {"Metrik": "ROUGE-L", "Skor (%)": global_rouge["rougeL"]},
        {"Metrik": "BERTScore Precision", "Skor": round(prec,4)},
        {"Metrik": "BERTScore Recall", "Skor": round(rec,4)},
        {"Metrik": "BERTScore F1", "Skor": round(f1,4)},
    ]).to_excel(writer, sheet_name="Ringkasan_Metrik", index=False)

    # Detail per klaster
    df_detail_sorted = df_detail.sort_values("rougeL", ascending=False)
    df_detail_sorted.to_excel(writer, sheet_name="Detail_Per_Klaster", index=False)

    # Sampel terbaik & terburuk
    terbaik = df_detail_sorted.head(5)[["cluster_id","rougeL","model_ringkasan","ref_ringkasan"]]
    terburuk = df_detail_sorted.tail(5)[["cluster_id","rougeL","model_ringkasan","ref_ringkasan"]]
    terbaik.to_excel(writer, sheet_name="Sampel_Terbaik", index=False)
    terburuk.to_excel(writer, sheet_name="Sampel_Terburuk", index=False)

    # Rumus metrik
    df_rumus.to_excel(writer, sheet_name="Rumus_Metrik", index=False)

    # Efisiensi komputasi
    df_efisiensi.to_excel(writer, sheet_name="Efisiensi_Komputasi", index=False)

    # Perbandingan
    df_comp.to_excel(writer, sheet_name="Perbandingan_BART_MiniLM", index=False)

files.download("/content/Laporan_Lengkap_BART.xlsx")
print("✅ Laporan lengkap dengan 7 sheet sudah diunduh!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Laporan lengkap dengan 7 sheet sudah diunduh!
